In [ ]:
from hcc_release_paths import project_path as _hcc_path
import pandas as pd
import numpy as np


clinical_df = pd.read_csv(
    _hcc_path('LIHC_clinical_clean.csv')
)


print(
    clinical_df.shape
)


clinical_df.head()

In [ ]:
clinical_clean = clinical_df[
    clinical_df["Tumor_Normal"]=="Tumor"
].copy()


print(
    clinical_clean.shape
)

In [ ]:
clinical_clean = (
    clinical_clean
    .drop_duplicates(
        subset="Patient_ID",
        keep="first"
    )
)


print(
    clinical_clean.shape
)

In [ ]:
clinical_clean["Gender_code"] = np.where(
    clinical_clean["Gender"]=="male",
    1,
    0
)

In [ ]:
clinical_clean["Stage_group"] = np.nan


clinical_clean.loc[
    clinical_clean["Stage"].isin(
        [
            "Stage I",
            "Stage IA",
            "Stage IB",
            "Stage II",
            "Stage IIA",
            "Stage IIB"
        ]
    ),
    "Stage_group"
] = 0



clinical_clean.loc[
    clinical_clean["Stage"].isin(
        [
            "Stage III",
            "Stage IIIA",
            "Stage IIIB",
            "Stage IIIC",
            "Stage IV",
            "Stage IVA",
            "Stage IVB"
        ]
    ),
    "Stage_group"
] = 1



clinical_clean["Stage_group"].value_counts()

In [ ]:
clinical_clean["OS"] = np.nan


clinical_clean.loc[
    clinical_clean["Vital_Status"]=="Alive",
    "OS"
]=0


clinical_clean.loc[
    clinical_clean["Vital_Status"]=="Dead",
    "OS"
]=1


clinical_clean["OS"].value_counts()

In [ ]:
clinical_clean.to_csv(
    "LIHC_clinical_clean_for_Cox.csv",
    index=False
)

In [ ]:
clinical_df.shape

In [ ]:
clinical_df.columns.tolist()

In [ ]:
# =====================================================
# Step9.1.1 Keep tumor samples
# =====================================================

clinical_clean = clinical_df[
    clinical_df["Tumor_Normal"]=="Tumor"
].copy()


print(
    "Tumor samples:",
    clinical_clean.shape
)

In [ ]:
# =====================================================
# Step9.1.2 Remove duplicated patients
# =====================================================


clinical_clean = (
    clinical_clean
    .drop_duplicates(
        subset="Patient_ID",
        keep="first"
    )
)


print(
    "Unique patients:",
    clinical_clean.shape
)

In [ ]:
# =====================================================
# Gender encoding
# =====================================================


clinical_clean["Gender_code"] = np.where(

    clinical_clean["Gender"]=="male",

    1,

    0

)


clinical_clean[
    "Gender_code"
].value_counts()

In [ ]:
# =====================================================
# Stage encoding
# =====================================================


clinical_clean["Stage_group"] = np.nan



clinical_clean.loc[

    clinical_clean["Stage"].isin(
        [
            "Stage I",
            "Stage IA",
            "Stage IB",
            "Stage II",
            "Stage IIA",
            "Stage IIB"
        ]
    ),

    "Stage_group"

] = 0




clinical_clean.loc[

    clinical_clean["Stage"].isin(
        [
            "Stage III",
            "Stage IIIA",
            "Stage IIIB",
            "Stage IIIC",
            "Stage IV",
            "Stage IVA",
            "Stage IVB"
        ]
    ),

    "Stage_group"

] = 1



print(
    clinical_clean["Stage_group"].value_counts(dropna=False)
)

In [ ]:
# =====================================================
# OS event encoding
# =====================================================


clinical_clean["OS"] = np.nan



clinical_clean.loc[
    clinical_clean["Vital_Status"]=="Alive",
    "OS"
]=0



clinical_clean.loc[
    clinical_clean["Vital_Status"]=="Dead",
    "OS"
]=1



print(
    clinical_clean["OS"].value_counts(dropna=False)
)

In [ ]:
# =====================================================
# Save cleaned clinical data
# =====================================================


clinical_clean.to_csv(
    "LIHC_clinical_clean_for_Cox.csv",
    index=False
)


print(
    "Saved!"
)

In [ ]:
clinical_clean[
[
"Patient_ID",
"Age",
"Gender",
"Gender_code",
"Stage",
"Stage_group",
"Vital_Status",
"OS"
]
].head()

In [ ]:
clinical_clean[
[
"Age",
"Gender_code",
"Stage_group",
"OS"
]
].isna().sum()

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =====================================================
# Step9.2 Load RiskScore
# =====================================================

risk_score_df = pd.read_csv(
    _hcc_path('LIHC_RNA_Step8_Final_Model/Results/Final_Top10_RiskScore.csv')
)


print(
    risk_score_df.shape
)


risk_score_df.head()

In [ ]:
risk_score_df["Patient_ID"] = (
    risk_score_df["Sample_ID"]
    .str.slice(0,12)
)

In [ ]:
risk_score_df[
[
"Sample_ID",
"Patient_ID",
"RiskScore"
]
].head()

In [ ]:
# =====================================================
# Merge clinical + RiskScore
# =====================================================


cox_data = clinical_clean.merge(

    risk_score_df,

    on="Patient_ID",

    how="inner"

)


print(
    cox_data.shape
)

In [ ]:
cox_data.columns.tolist()

In [ ]:
# =====================================================
# Step9.2.1 Clean merged Cox dataset
# =====================================================


cox_data = cox_data.rename(
    columns={
        "OS_y":"OS",
        "Sample_ID_y":"Risk_Sample_ID"
    }
)


# 删除重复变量

cox_data = cox_data.drop(
    columns=[
        "OS_x",
        "Sample_ID_x"
    ],
    errors="ignore"
)



print(
    cox_data.shape
)


cox_data.head()

In [ ]:
cox_data[
[
"Patient_ID",
"RiskScore",
"OS_time",
"OS",
"Age",
"Gender_code",
"Stage_group"
]
].head()

In [ ]:
print(
    "Patients:",
    cox_data["Patient_ID"].nunique()
)

In [ ]:
cox_data[
[
"RiskScore",
"OS_time",
"OS",
"Age",
"Gender_code",
"Stage_group"
]
].isna().sum()

In [ ]:
# =====================================================
# Step9.3 Univariate Cox analysis
# =====================================================

from lifelines import CoxPHFitter
import pandas as pd
import numpy as np


uni_results = []


variables = [
    "RiskScore",
    "Age",
    "Gender_code",
    "Stage_group"
]


for var in variables:

    cph = CoxPHFitter()


    data = cox_data[
        [
            "OS_time",
            "OS",
            var
        ]
    ].dropna()


    cph.fit(
        data,
        duration_col="OS_time",
        event_col="OS"
    )


    summary = cph.summary.loc[var]


    uni_results.append({

        "Variable":var,

        "HR":summary["exp(coef)"],

        "HR_lower95":
        summary["exp(coef) lower 95%"],

        "HR_upper95":
        summary["exp(coef) upper 95%"],

        "P_value":
        summary["p"]

    })


uni_cox = pd.DataFrame(
    uni_results
)


uni_cox

In [ ]:
uni_cox.to_csv(
    "Univariate_Cox_results.csv",
    index=False
)

In [ ]:
# =====================================================
# Step9.4 Multivariate Cox
# =====================================================


from lifelines import CoxPHFitter


multi_data = cox_data[
[
"OS_time",
"OS",
"RiskScore",
"Age",
"Gender_code",
"Stage_group"
]
].dropna()



print(
    multi_data.shape
)

In [ ]:
# =====================================================
# Multivariate Cox model
# =====================================================


cph_multi = CoxPHFitter()


cph_multi.fit(

    multi_data,

    duration_col="OS_time",

    event_col="OS"

)


multi_summary = cph_multi.summary


multi_summary

In [ ]:
# =====================================================
# Export multivariate Cox table
# =====================================================


multi_cox = pd.DataFrame({

    "Variable":
    multi_summary.index,


    "HR":
    multi_summary["exp(coef)"],


    "HR_lower95":
    multi_summary["exp(coef) lower 95%"],


    "HR_upper95":
    multi_summary["exp(coef) upper 95%"],


    "P_value":
    multi_summary["p"]

})


multi_cox

In [ ]:
multi_cox.to_csv(
    "Multivariate_Cox_results.csv",
    index=False
)

In [ ]:
import pandas as pd
import numpy as np
import os

# 如果 multi_cox 已经存在，就直接用
# 如果没有，可以先看看：
print(multi_cox.head())
print(multi_cox.columns)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# ============================================
# Step9.5 Forest plot for multivariate Cox
# ============================================

# 输出文件夹
step9_dir = "LIHC_RNA_Step9"
os.makedirs(step9_dir, exist_ok=True)

# 复制一份，避免改原表
forest_df = multi_cox.copy()

# 只保留想展示的变量，并按想要顺序排序
var_order = ["RiskScore", "Stage_group", "Age", "Gender_code"]
forest_df = forest_df[forest_df["Variable"].isin(var_order)].copy()

forest_df["Variable"] = pd.Categorical(
    forest_df["Variable"],
    categories=var_order,
    ordered=True
)

forest_df = forest_df.sort_values("Variable").reset_index(drop=True)

# 变量名美化
label_map = {
    "RiskScore": "Risk score",
    "Stage_group": "Stage (III-IV vs I-II)",
    "Age": "Age",
    "Gender_code": "Gender (male vs female)"
}

forest_df["Label"] = forest_df["Variable"].map(label_map)

# P值格式化
def format_p(p):
    if p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"

forest_df["P_text"] = forest_df["P_value"].apply(format_p)

# HR(95%CI)格式化
forest_df["HR_CI"] = forest_df.apply(
    lambda x: f'{x["HR"]:.3f} ({x["HR_lower95"]:.3f}-{x["HR_upper95"]:.3f})',
    axis=1
)

# y轴位置（让RiskScore显示在最上面）
forest_df = forest_df.iloc[::-1].reset_index(drop=True)
y_pos = np.arange(len(forest_df))

# 开始画图
fig, ax = plt.subplots(figsize=(10, 4.8))

# 误差线
ax.errorbar(
    forest_df["HR"],
    y_pos,
    xerr=[
        forest_df["HR"] - forest_df["HR_lower95"],
        forest_df["HR_upper95"] - forest_df["HR"]
    ],
    fmt="o",
    capsize=4,
    linewidth=2
)

# 参考线 HR=1
ax.axvline(
    1,
    linestyle="--",
    linewidth=1
)

# y轴标签
ax.set_yticks(y_pos)
ax.set_yticklabels(forest_df["Label"], fontsize=11)

# x轴
ax.set_xlabel("Hazard ratio (95% CI)", fontsize=11)
ax.set_xscale("log")
ax.set_xlim(0.5, 4.5)
ax.set_xticks([0.5, 1, 2, 3, 4])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

# 图标题
ax.set_title("Multivariate Cox Regression Analysis", fontsize=13)

# 去掉多余边框
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 在右侧添加 HR(95%CI) 和 P value
trans = ax.get_yaxis_transform()

for i, row in forest_df.iterrows():
    ax.text(
        1.03, i,
        row["HR_CI"],
        transform=trans,
        va="center",
        fontsize=10,
        clip_on=False
    )
    ax.text(
        1.48, i,
        row["P_text"],
        transform=trans,
        va="center",
        fontsize=10,
        clip_on=False
    )

# 列标题
ax.text(
    1.03, len(forest_df)-0.2,
    "HR (95% CI)",
    transform=trans,
    fontsize=11,
    fontweight="bold",
    clip_on=False
)
ax.text(
    1.48, len(forest_df)-0.2,
    "P value",
    transform=trans,
    fontsize=11,
    fontweight="bold",
    clip_on=False
)

plt.tight_layout()

# 一定要先保存，再show，避免白图
png_path = os.path.join(step9_dir, "01_Multivariate_Cox_ForestPlot.png")
pdf_path = os.path.join(step9_dir, "01_Multivariate_Cox_ForestPlot.pdf")

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
# =====================================================
# Final Cox Forest Plot
# Univariate + Multivariate
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


# -----------------------------
# prepare data
# -----------------------------

variables = [
    "RiskScore",
    "Stage_group",
    "Age",
    "Gender_code"
]


label_map = {

    "RiskScore":
    "Risk score (10-gene signature)",

    "Stage_group":
    "Stage (III-IV vs I-II)",

    "Age":
    "Age",

    "Gender_code":
    "Gender (male vs female)"

}


# 保证顺序

uni = uni_cox[
    uni_cox["Variable"].isin(variables)
].copy()


multi = multi_cox[
    multi_cox["Variable"].isin(variables)
].copy()



uni["order"] = uni["Variable"].map(
    {v:i for i,v in enumerate(variables)}
)

multi["order"] = multi["Variable"].map(
    {v:i for i,v in enumerate(variables)}
)


uni = uni.sort_values("order")
multi = multi.sort_values("order")



# -----------------------------
# format text
# -----------------------------

def format_hr(row):

    return (
        f'{row["HR"]:.2f}'
        f' ({row["HR_lower95"]:.2f}-'
        f'{row["HR_upper95"]:.2f})'
    )


def format_p(p):

    if p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"



uni["HR_text"] = uni.apply(
    format_hr,
    axis=1
)

multi["HR_text"] = multi.apply(
    format_hr,
    axis=1
)


uni["P_text"] = uni["P_value"].apply(
    format_p
)

multi["P_text"] = multi["P_value"].apply(
    format_p
)



# -----------------------------
# plotting function
# -----------------------------

def draw_forest(ax, df, title):

    y = np.arange(len(df))


    ax.errorbar(

        df["HR"],

        y,

        xerr=[

            df["HR"] -
            df["HR_lower95"],


            df["HR_upper95"] -
            df["HR"]

        ],

        fmt="o",

        markersize=7,

        capsize=4,

        linewidth=2

    )


    ax.axvline(

        1,

        linestyle="--",

        linewidth=1

    )


    ax.set_xscale("log")


    ax.set_xlim(
        0.5,
        4
    )


    ax.set_xticks(
        [
            0.5,
            1,
            2,
            3,
            4
        ]
    )


    ax.set_xticklabels(
        [
            "0.5",
            "1",
            "2",
            "3",
            "4"
        ]
    )


    ax.set_yticks(y)

    ax.set_yticklabels(
        [
            label_map[x]
            for x in df["Variable"]
        ],
        fontsize=10
    )


    ax.set_xlabel(
        "Hazard ratio (95% CI)",
        fontsize=11
    )


    ax.set_title(
        title,
        fontsize=13,
        fontweight="bold"
    )


    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)



# -----------------------------
# create figure
# -----------------------------

fig, axes = plt.subplots(

    1,

    2,

    figsize=(13,5),

    sharey=True

)



draw_forest(

    axes[0],

    uni,

    "Univariate Cox regression"

)


draw_forest(

    axes[1],

    multi,

    "Multivariate Cox regression"

)



# -----------------------------
# add HR text
# -----------------------------

for ax,df in zip(
    axes,
    [uni,multi]
):

    for i,row in df.iterrows():

        ax.text(

            1.02,

            row["order"],

            row["HR_text"],

            transform=ax.get_yaxis_transform(),

            va="center",

            fontsize=9

        )


        ax.text(

            1.45,

            row["order"],

            row["P_text"],

            transform=ax.get_yaxis_transform(),

            va="center",

            fontsize=9

        )



    ax.text(

        1.02,

        len(df),

        "HR (95% CI)",

        transform=ax.get_yaxis_transform(),

        fontsize=9,

        fontweight="bold"

    )


    ax.text(

        1.45,

        len(df),

        "P",

        transform=ax.get_yaxis_transform(),

        fontsize=9,

        fontweight="bold"

    )



plt.tight_layout()



# -----------------------------
# save
# -----------------------------

outdir = "LIHC_Final_Figures"

os.makedirs(
    outdir,
    exist_ok=True
)


plt.savefig(

    os.path.join(
        outdir,
        "Fig_Cox_Univariate_Multivariate_Forest.png"
    ),

    dpi=600,

    bbox_inches="tight"

)


plt.savefig(

    os.path.join(
        outdir,
        "Fig_Cox_Univariate_Multivariate_Forest.pdf"
    ),

    bbox_inches="tight"

)


plt.show()

In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter


nomogram_data = multi_data[
[
"OS_time",
"OS",
"RiskScore",
"Stage_group"
]
].copy()


print(nomogram_data.shape)

In [ ]:
cph_nom = CoxPHFitter()


cph_nom.fit(

    nomogram_data,

    duration_col="OS_time",

    event_col="OS"

)


cph_nom.summary

In [ ]:
# 时间点
times = [
365,
1095,
1825
]


surv_prob = cph_nom.predict_survival_function(
    nomogram_data.drop(
        columns=["OS_time","OS"]
    ),
    times=times
)


surv_prob.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================
# Extract Cox coefficients
# ============================

beta = cph_nom.params_


risk_beta = beta["RiskScore"]
stage_beta = beta["Stage_group"]


print(beta)



# ============================
# Define point scale
# ============================

max_points = 100


# RiskScore range

risk_min = nomogram_data["RiskScore"].min()

risk_max = nomogram_data["RiskScore"].max()



# Stage

stage_values = [0,1]



# 最大贡献对应100分

risk_points_max = 100


# coefficient contribution

risk_range = (
    risk_max-risk_min
)*risk_beta


stage_range = (
    max(stage_values)-min(stage_values)
)*stage_beta


total_range=max(
abs(risk_range),
abs(stage_range)
)


risk_scale = (
100*abs(risk_beta)
/
total_range
)


stage_scale = (
100*abs(stage_beta)
/
total_range
)


print(
risk_scale,
stage_scale
)

In [ ]:
import numpy as np
import pandas as pd


# ==========================
# Predict risk probability
# ==========================

times = [
    365,
    1095,
    1825
]


risk_predictions = {}


X = nomogram_data[
[
"RiskScore",
"Stage_group"
]
]


for t in times:

    surv = cph_nom.predict_survival_function(
        X,
        times=[t]
    ).T.iloc[:,0]


    risk_predictions[
        f"Risk_{t//365}y"
    ] = 1-surv



risk_df = pd.DataFrame(
    risk_predictions
)


risk_df.head()

In [ ]:
def decision_curve(
    y_true,
    y_prob,
    thresholds=np.arange(0.01,0.99,0.01)
):

    N=len(y_true)

    results=[]


    for pt in thresholds:


        pred_positive = y_prob >= pt


        TP=np.sum(
            pred_positive & (y_true==1)
        )


        FP=np.sum(
            pred_positive & (y_true==0)
        )


        net_benefit = (
            TP/N
            -
            FP/N *
            (pt/(1-pt))
        )


        results.append(
            [
                pt,
                net_benefit
            ]
        )


    return pd.DataFrame(
        results,
        columns=[
            "threshold",
            "net_benefit"
        ]
    )

In [ ]:
plt.figure(figsize=(7,6))


colors=[
"1-year",
"3-year",
"5-year"
]


for col,label in zip(
    [
    "Risk_1y",
    "Risk_3y",
    "Risk_5y"
    ],
    colors
):

    if col=="Risk_1y":
        t=365
    elif col=="Risk_3y":
        t=1095
    else:
        t=1825


    y=(
        (nomogram_data["OS_time"]<=t)
        &
        (nomogram_data["OS"]==1)
    ).astype(int)



    dca=decision_curve(
        y.values,
        risk_df[col].values
    )


    plt.plot(
        dca["threshold"],
        dca["net_benefit"],
        linewidth=2,
        label=label
    )


plt.axhline(
    0,
    linestyle="--"
)


plt.xlabel(
"Threshold probability"
)


plt.ylabel(
"Net benefit"
)


plt.legend()


plt.title(
"Decision curve analysis"
)


plt.savefig(
"DCA_all_timepoints.png",
dpi=600,
bbox_inches="tight"
)


plt.show()

In [ ]:
# ============================================================
# Step10 DCA
# Basic functions
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 输出目录
dca_dir = os.path.join(
    "LIHC_RNA_Step10_DCA"
)

os.makedirs(
    dca_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# Decision Curve Analysis
# ------------------------------------------------------------

def calculate_dca(
    y_true,
    y_prob,
    thresholds
):

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    n = len(y_true)

    model_nb = []
    all_nb = []
    none_nb = []

    prevalence = y_true.mean()

    for pt in thresholds:

        predicted_positive = (
            y_prob >= pt
        )

        tp = np.sum(
            predicted_positive &
            (y_true == 1)
        )

        fp = np.sum(
            predicted_positive &
            (y_true == 0)
        )

        # Model net benefit
        nb_model = (
            tp / n
            -
            fp / n *
            pt / (1 - pt)
        )

        # Treat-all net benefit
        nb_all = (
            prevalence
            -
            (1 - prevalence) *
            pt / (1 - pt)
        )

        # Treat-none
        nb_none = 0.0

        model_nb.append(nb_model)
        all_nb.append(nb_all)
        none_nb.append(nb_none)

    return pd.DataFrame({
        "Threshold": thresholds,
        "Model": model_nb,
        "Treat_all": all_nb,
        "Treat_none": none_nb
    })

In [ ]:
# ============================================================
# Predicted risk at 1 / 3 / 5 years
# ============================================================

time_points = {
    "1-year": 365,
    "3-year": 1095,
    "5-year": 1825
}


X_dca = nomogram_data[
    [
        "RiskScore",
        "Stage_group"
    ]
].copy()


predicted_risk = {}


for label, t in time_points.items():

    survival_prob = (
        cph_nom
        .predict_survival_function(
            X_dca,
            times=[t]
        )
        .T
        .iloc[:, 0]
    )

    predicted_risk[label] = (
        1 - survival_prob
    )


risk_dca = pd.DataFrame(
    predicted_risk,
    index=nomogram_data.index
)


display(
    risk_dca.head()
)

In [ ]:
# ============================================================
# Thresholds
# ============================================================

thresholds = np.arange(
    0.01,
    0.61,
    0.01
)

In [ ]:
# ============================================================
# 1-year DCA
# ============================================================

t = 365


y_1y = (
    (
        nomogram_data["OS_time"] <= t
    )
    &
    (
        nomogram_data["OS"] == 1
    )
).astype(int)


dca_1y = calculate_dca(
    y_true=y_1y,
    y_prob=risk_dca["1-year"],
    thresholds=thresholds
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


ax.plot(
    dca_1y["Threshold"],
    dca_1y["Model"],
    linewidth=2.2,
    label="Prognostic model"
)


ax.plot(
    dca_1y["Threshold"],
    dca_1y["Treat_all"],
    linestyle="--",
    linewidth=1.6,
    label="Treat all"
)


ax.plot(
    dca_1y["Threshold"],
    dca_1y["Treat_none"],
    linestyle=":",
    linewidth=1.6,
    label="Treat none"
)


ax.set_xlim(
    0,
    0.60
)


ax.set_xlabel(
    "Threshold probability",
    fontsize=12
)


ax.set_ylabel(
    "Net benefit",
    fontsize=12
)


ax.set_title(
    "Decision curve analysis for 1-year overall survival",
    fontsize=13
)


ax.legend(
    frameon=False,
    fontsize=10
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


# 先保存
fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_1year_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_1year_OS.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# 3-year DCA
# ============================================================

t = 1095


y_3y = (
    (
        nomogram_data["OS_time"] <= t
    )
    &
    (
        nomogram_data["OS"] == 1
    )
).astype(int)


dca_3y = calculate_dca(
    y_true=y_3y,
    y_prob=risk_dca["3-year"],
    thresholds=thresholds
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


ax.plot(
    dca_3y["Threshold"],
    dca_3y["Model"],
    linewidth=2.2,
    label="Prognostic model"
)


ax.plot(
    dca_3y["Threshold"],
    dca_3y["Treat_all"],
    linestyle="--",
    linewidth=1.6,
    label="Treat all"
)


ax.plot(
    dca_3y["Threshold"],
    dca_3y["Treat_none"],
    linestyle=":",
    linewidth=1.6,
    label="Treat none"
)


ax.set_xlim(
    0,
    0.60
)


ax.set_xlabel(
    "Threshold probability",
    fontsize=12
)


ax.set_ylabel(
    "Net benefit",
    fontsize=12
)


ax.set_title(
    "Decision curve analysis for 3-year overall survival",
    fontsize=13
)


ax.legend(
    frameon=False,
    fontsize=10
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_3year_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_3year_OS.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# 5-year DCA
# ============================================================

t = 1825


y_5y = (
    (
        nomogram_data["OS_time"] <= t
    )
    &
    (
        nomogram_data["OS"] == 1
    )
).astype(int)


dca_5y = calculate_dca(
    y_true=y_5y,
    y_prob=risk_dca["5-year"],
    thresholds=thresholds
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


ax.plot(
    dca_5y["Threshold"],
    dca_5y["Model"],
    linewidth=2.2,
    label="Prognostic model"
)


ax.plot(
    dca_5y["Threshold"],
    dca_5y["Treat_all"],
    linestyle="--",
    linewidth=1.6,
    label="Treat all"
)


ax.plot(
    dca_5y["Threshold"],
    dca_5y["Treat_none"],
    linestyle=":",
    linewidth=1.6,
    label="Treat none"
)


ax.set_xlim(
    0,
    0.60
)


ax.set_xlabel(
    "Threshold probability",
    fontsize=12
)


ax.set_ylabel(
    "Net benefit",
    fontsize=12
)


ax.set_title(
    "Decision curve analysis for 5-year overall survival",
    fontsize=13
)


ax.legend(
    frameon=False,
    fontsize=10
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_5year_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_5year_OS.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# Combined DCA
# 1 / 3 / 5 years
# ============================================================


fig, ax = plt.subplots(
    figsize=(8, 6)
)


ax.plot(
    dca_1y["Threshold"],
    dca_1y["Model"],
    linewidth=2.2,
    label="1-year model"
)


ax.plot(
    dca_3y["Threshold"],
    dca_3y["Model"],
    linewidth=2.2,
    label="3-year model"
)


ax.plot(
    dca_5y["Threshold"],
    dca_5y["Model"],
    linewidth=2.2,
    label="5-year model"
)


# Treat none
ax.axhline(
    y=0,
    linestyle="--",
    linewidth=1.4,
    label="Treat none"
)


ax.set_xlim(
    0,
    0.60
)


ax.set_xlabel(
    "Threshold probability",
    fontsize=12
)


ax.set_ylabel(
    "Net benefit",
    fontsize=12
)


ax.set_title(
    "Decision curve analysis of the prognostic model",
    fontsize=13
)


ax.legend(
    frameon=False,
    fontsize=10
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_Combined_1_3_5year.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        dca_dir,
        "DCA_Combined_1_3_5year.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
dca_1y.to_csv(
    os.path.join(
        dca_dir,
        "DCA_1year_results.csv"
    ),
    index=False
)


dca_3y.to_csv(
    os.path.join(
        dca_dir,
        "DCA_3year_results.csv"
    ),
    index=False
)


dca_5y.to_csv(
    os.path.join(
        dca_dir,
        "DCA_5year_results.csv"
    ),
    index=False
)


print("All DCA figures and tables saved.")

In [ ]:
# ============================================================
# Step10.2 Survival Calibration
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter


cal_dir = "LIHC_RNA_Step10_Calibration"

os.makedirs(
    cal_dir,
    exist_ok=True
)

In [ ]:
# ============================================================
# Predicted survival probability
# ============================================================

X_cal = nomogram_data[
    [
        "RiskScore",
        "Stage_group"
    ]
].copy()


time_points = {
    "1-year": 365,
    "3-year": 1095,
    "5-year": 1825
}


pred_survival = pd.DataFrame(
    index=nomogram_data.index
)


for label, t in time_points.items():

    pred = (
        cph_nom
        .predict_survival_function(
            X_cal,
            times=[t]
        )
        .T
        .iloc[:, 0]
    )

    pred_survival[label] = pred


display(
    pred_survival.head()
)

In [ ]:
# ============================================================
# Survival calibration function
# Kaplan-Meier observed survival
# ============================================================

def survival_calibration(
    data,
    predicted_survival,
    time_point,
    n_bins=5
):

    temp = pd.DataFrame({

        "OS_time":
        data["OS_time"].values,

        "OS":
        data["OS"].values,

        "Predicted":
        np.asarray(predicted_survival)

    })


    # 按预测生存概率分组
    temp["Bin"] = pd.qcut(
        temp["Predicted"],
        q=n_bins,
        duplicates="drop"
    )


    results = []


    for bin_name, group in temp.groupby(
        "Bin",
        observed=False
    ):

        # 平均预测生存率
        mean_pred = group[
            "Predicted"
        ].mean()


        # KM估计实际生存率
        kmf = KaplanMeierFitter()

        kmf.fit(
            durations=group["OS_time"],
            event_observed=group["OS"]
        )


        observed_survival = (
            kmf.predict(time_point)
        )


        results.append({

            "Predicted_survival":
            mean_pred,

            "Observed_survival":
            observed_survival,

            "N":
            len(group)

        })


    result_df = pd.DataFrame(
        results
    )


    return result_df.sort_values(
        "Predicted_survival"
    )

In [ ]:
# ============================================================
# Calculate calibration
# ============================================================

calibration_results = {}


for label, t in time_points.items():

    calibration_results[label] = (
        survival_calibration(
            data=nomogram_data,
            predicted_survival=
                pred_survival[label],
            time_point=t,
            n_bins=5
        )
    )


    print("\n", label)

    display(
        calibration_results[label]
    )

In [ ]:
# ============================================================
# 1-year Calibration
# ============================================================

cal = calibration_results[
    "1-year"
]


fig, ax = plt.subplots(
    figsize=(6, 6)
)


# Perfect calibration
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Ideal"
)


# Model
ax.plot(
    cal["Predicted_survival"],
    cal["Observed_survival"],
    marker="o",
    markersize=7,
    linewidth=2,
    label="Prognostic model"
)


ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1
)


ax.set_xlabel(
    "Predicted 1-year survival probability",
    fontsize=11
)


ax.set_ylabel(
    "Observed 1-year survival probability",
    fontsize=11
)


ax.set_title(
    "Calibration curve for 1-year overall survival",
    fontsize=13
)


ax.legend(
    frameon=False
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_1year_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_1year_OS.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# 3-year Calibration
# ============================================================

cal = calibration_results[
    "3-year"
]


fig, ax = plt.subplots(
    figsize=(6, 6)
)


ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Ideal"
)


ax.plot(
    cal["Predicted_survival"],
    cal["Observed_survival"],
    marker="o",
    markersize=7,
    linewidth=2,
    label="Prognostic model"
)


ax.set_xlim(0, 1)
ax.set_ylim(0, 1)


ax.set_xlabel(
    "Predicted 3-year survival probability",
    fontsize=11
)


ax.set_ylabel(
    "Observed 3-year survival probability",
    fontsize=11
)


ax.set_title(
    "Calibration curve for 3-year overall survival",
    fontsize=13
)


ax.legend(
    frameon=False
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_3year_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_3year_OS.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# 5-year Calibration
# ============================================================

cal = calibration_results[
    "5-year"
]


fig, ax = plt.subplots(
    figsize=(6, 6)
)


ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Ideal"
)


ax.plot(
    cal["Predicted_survival"],
    cal["Observed_survival"],
    marker="o",
    markersize=7,
    linewidth=2,
    label="Prognostic model"
)


ax.set_xlim(0, 1)
ax.set_ylim(0, 1)


ax.set_xlabel(
    "Predicted 5-year survival probability",
    fontsize=11
)


ax.set_ylabel(
    "Observed 5-year survival probability",
    fontsize=11
)


ax.set_title(
    "Calibration curve for 5-year overall survival",
    fontsize=13
)


ax.legend(
    frameon=False
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_5year_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_5year_OS.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# Combined Calibration
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)


# Ideal line
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Ideal"
)


for label in [
    "1-year",
    "3-year",
    "5-year"
]:

    cal = calibration_results[
        label
    ]


    ax.plot(
        cal["Predicted_survival"],
        cal["Observed_survival"],
        marker="o",
        markersize=6,
        linewidth=2,
        label=label
    )


ax.set_xlim(
    0,
    1
)


ax.set_ylim(
    0,
    1
)


ax.set_xlabel(
    "Predicted survival probability",
    fontsize=12
)


ax.set_ylabel(
    "Observed survival probability",
    fontsize=12
)


ax.set_title(
    "Calibration of the prognostic model",
    fontsize=13
)


ax.legend(
    frameon=False
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_Combined_1_3_5year.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        cal_dir,
        "Calibration_Combined_1_3_5year.pdf"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
for label, cal in calibration_results.items():

    name = label.replace(
        "-",
        "_"
    )


    cal.to_csv(
        os.path.join(
            cal_dir,
            f"Calibration_{name}_results.csv"
        ),
        index=False
    )


print(
    "All calibration figures and results saved."
)